# YOLO11m + temporal post-processing (Kaggle or Colab, ~3 h T4)

VisDrone frames come from videos: an object seen in frame t−1 is almost always still there in frame t.
A single-frame detector ignores this, so small objects flicker between a score of 0.4 and 0.05, or vanish
for a frame or two under blur or occlusion. This notebook trains **YOLO11m** (the larger model the YOLO11s
search pointed to) and then improves its per-frame detections with the past frames of the same clip
(`src.analysis.temporal`, causal: only earlier frames are used):

1. **Camera motion** (`motion`): a similarity transform between consecutive frames from ORB keypoints +
   RANSAC; track boxes are warped before matching, otherwise a panning drone moves 10 px objects out of IoU range.
2. **Tracking**: per class, tracks are matched to the current detections by IoU (Hungarian).
3. **Rescoring** (`weight`): a matched detection's score is blended with its track's running score:
   weak detections of objects seen confidently before go up, one-frame false positives go down.
4. **Propagation** (`max_age`): a confident track missed in this frame adds its warped box with a decayed
   score for up to `max_age` frames.

**Protocol.** YOLO11m is trained on every 5th train frame at `imgsz` 1280. The first 12-epoch run
(`yolo11m_1280`, mosaic in epochs 1–10) peaked at epoch 3 and then overfit: its train cls_loss kept falling
while the val cls_loss rose from epoch 5. This version therefore trains **6 epochs** (the learning rate decays
before the overfitting starts), keeps mosaic for the first 2 epochs only (the better setting of the section-1
check), and adds stronger augmentation (`mixup` 0.1, `copy_paste` 0.1, `scale` 0.7). Its standard result is
the usual every-5th-val-frame COCO evaluation. The **submission model** is then retrained with the same
settings on **train + val** (section 6): the val clips are scenes the train split does not have.
Tracking needs consecutive frames, so the temporal step runs on **all** val frames. Its parameters are tuned
on 4 of the 7 val clips and the gain is reported on the other **3 held-out clips**, so the reported gain is
not fitted to the frames it is measured on. COCO AP with maxDets 500, broken down by object size.

Runs on **Kaggle** (GPU T4, Internet on, competition data via **Add Input**, then **Save & Run All**) or
**Colab** (T4; secrets `KAGGLE_API_TOKEN` or `KAGGLE_USERNAME` + `KAGGLE_KEY` for the data, `WANDB_API_KEY`
for W&B). On Colab the run lives on Google Drive: after a disconnect **Run all** again and training resumes.

In [ ]:
import os, sys, time, re
from pathlib import Path

T0 = time.time()
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # set only on Kaggle; Colab can also have /kaggle/input
WORK = Path("/kaggle/working" if ON_KAGGLE else "/content")
REPO = WORK / "repo"
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
!nvidia-smi -L
![ -d {REPO}/.git ] && git -C {REPO} pull -q || git clone -q https://github.com/LiliAlania/drone-multi-object-detection.git {REPO}
%cd {REPO}
!pip install -q -U kaggle  # Colab's preinstalled kaggle CLI may not know the new API tokens
!pip install -q ultralytics==8.4.208 pycocotools wandb
sys.path.insert(0, str(REPO))
# the code comes from GitHub, not from this notebook: stop now if it predates the src/ restructure
assert (REPO / "src/analysis/temporal.py").is_file(), \
    "GitHub has an older version of the repo: commit and push before running this notebook"

In [ ]:
MODEL = "yolo11m.pt"
IMGSZ = 1280          # best resolution of the YOLO11s search (notebook 05)
# the 12-epoch yolo11m_1280 run peaked at epoch 3 (Ultralytics val mAP50-95 0.238, 0.213 at epoch 12) and its
# val cls_loss rose from epoch 5: fewer epochs, so the learning rate decays before the overfitting starts
EPOCHS = 6
# mosaic off for the last CLOSE_MOSAIC epochs = mosaic in epochs 1-2, the better setting of section 1
# (close_mosaic >= EPOCHS would switch mosaic off from the first epoch)
CLOSE_MOSAIC = 4
AUG = {"mixup": 0.1, "copy_paste": 0.1, "scale": 0.7}  # against the class overfitting; Ultralytics: 0, 0, 0.5
FINAL_TRAIN_VAL = True  # section 6: retrain with the same settings on train + val for the submission (~1.2 h)
MOSAIC_CHECK = False  # section 1, done (yolo_runs/mosaic_check.png): YOLO11s at 960, 12 epochs (~45 min per run)
BATCH = 4            # Ultralytics accumulates to a nominal batch of 64; halve if CUDA runs out of memory
SUBSET_STEP = 5       # every 5th train frame, as every other detector
EVAL_STEP = 5         # standard evaluation: every 5th val frame
WORKERS = 4 if ON_KAGGLE else 2
NAME = f"yolo11m_{IMGSZ}_e{EPOCHS}_aug"  # new folder: nothing reused from the 12-epoch run
RUNS = Path("/kaggle/working/yolo_runs") if ON_KAGGLE else Path("/content/drive/MyDrive/yolo_runs")
RUN = RUNS / NAME
WEIGHTS = RUN / "weights" / "last.pt"  # final-epoch weights, as in notebook 05
# to skip training and use an existing model, point WEIGHTS at it, e.g.
# WEIGHTS = RUNS / "yolo11s_p2_1280" / "weights" / "last.pt"
RUNS.mkdir(parents=True, exist_ok=True)

# W&B: a *team* entity you can write to (an organization name is refused); "" = the key's default entity
WANDB_PROJECT = "drone-multi-object-detection"
WANDB_ENTITY = ""
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_DIR"] = str(RUNS)  # offline runs then land next to the checkpoints


def secret(name):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception as error:  # Colab: NotebookAccessError = the secret's "Notebook access" toggle is off
        if type(error).__name__ != "SecretNotFoundError":
            print(f"cannot read secret {name}: {type(error).__name__}: {error}")
        return None


for name in ["WANDB_API_KEY"] + ([] if ON_KAGGLE else ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]):
    value = secret(name)
    if value:
        os.environ[name] = value
if os.environ.get("KAGGLE_API_TOKEN"):
    token_file = Path.home() / ".kaggle" / "access_token"
    token_file.parent.mkdir(exist_ok=True)
    token_file.write_text(os.environ["KAGGLE_API_TOKEN"])
    token_file.chmod(0o600)
if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):  # the new CLI reads the legacy key only from kaggle.json
    import json
    legacy_file = Path.home() / ".kaggle" / "kaggle.json"
    legacy_file.parent.mkdir(exist_ok=True)
    legacy_file.write_text(json.dumps({"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}))
    legacy_file.chmod(0o600)
HAVE_KAGGLE_KEY = ON_KAGGLE or bool(os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY"))
if not HAVE_KAGGLE_KEY:
    print("no Kaggle credentials: add the Colab secret KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY) "
          "with Notebook access on, then rerun this cell")
if "WANDB_API_KEY" in os.environ:
    os.environ.pop("WANDB_MODE", None)  # an earlier run of this cell may have switched W&B offline
else:
    print("no WANDB_API_KEY secret: W&B logs offline")
    os.environ["WANDB_MODE"] = "offline"
print("platform:", "Kaggle" if ON_KAGGLE else "Colab", "| W&B:", os.environ.get("WANDB_MODE", "online"))

In [ ]:
# competition data: Kaggle input, or downloaded once per Colab session to local disk
INPUT = Path("/kaggle/input") if ON_KAGGLE else Path("/content/data")
find = lambda: [p for depth in range(0, 5) for p in INPUT.glob("*/" * depth + "val/annotations_coco.json")]
if not find() and not ON_KAGGLE:
    assert HAVE_KAGGLE_KEY, "add the Kaggle secret(s) in Colab (see the config cell output) and rerun the config cell"
    !python -m src.data.get_data --output-dir {INPUT}
found = find()
assert found, "Competition data not found: add it with Add Input (Kaggle) or set the Kaggle secrets (Colab)"
VAL_ROOT = found[0].parent
DATA = VAL_ROOT.parent
print("data:", DATA, sorted(p.name for p in DATA.iterdir()))

SUBSET = WORK / "visdrone-subset"
if not (SUBSET / "images" / "train").is_dir():
    !python -m src.data.make_subset --data-root {DATA} --output-dir {SUBSET} --step {SUBSET_STEP} --splits train val
    !rm -f {SUBSET}.zip
print(f"{len(list((SUBSET / 'images' / 'train').rglob('*.jpg')))} train images, "
      f"{(time.time() - T0) / 60:.0f} min elapsed")

In [ ]:
import pandas as pd

METRIC = re.compile(r"mAP@\[\.5:\.95\] ([\d.]+) \| AP50 ([\d.]+) \| AP_small ([\d.]+) \| "
                    r"AP_medium ([\d.]+) \| AP_large ([\d.]+)")


def metrics(name):
    path = RUNS / name / "eval.txt"
    found = METRIC.search(path.read_text(errors="ignore")) if path.is_file() else None
    return dict(zip(["mAP", "AP50", "AP_small", "AP_medium", "AP_large"], map(float, found.groups()))) if found else None


def finished(name, epochs):
    path = RUNS / name / "results.csv"  # Ultralytics adds one row per finished epoch
    return path.is_file() and len(path.read_text().strip().splitlines()) > epochs


def train_and_evaluate(name, model, imgsz, batch, close_mosaic, epochs=EPOCHS, flags="", evaluate=True):
    # steps finished in an earlier session are skipped; training resumes from last.pt after a disconnect
    last = RUNS / name / "weights" / "last.pt"
    if finished(name, epochs) or metrics(name):
        print(f"{name}: trained in an earlier session")
    else:
        resume = "--resume" if last.is_file() else ""
        print(f"=== {name}: {model}, imgsz {imgsz}, {epochs} epochs, mosaic off for the last {close_mosaic} {flags} ===")
        get_ipython().system(  # what `!` runs; called directly so it works inside a function
            f"python -m src.models.yolo_baseline train --data-root {SUBSET} --model {model} --epochs {epochs} "
            f"--imgsz {imgsz} --batch {batch} --close-mosaic {close_mosaic} --workers {WORKERS} --project {RUNS} "
            f"--name {name} {flags} {resume} --wandb-project {WANDB_PROJECT}")
    if not evaluate or metrics(name):
        return
    get_ipython().system(
        f"python -m src.models.yolo_baseline evaluate --weights {last} --split-root {VAL_ROOT} --step {EVAL_STEP} "
        f"--imgsz {imgsz} --wandb-project {WANDB_PROJECT} | tee {RUNS / name}/eval.txt")
    print(f"{(time.time() - T0) / 60:.0f} min elapsed")

## 1. Does mosaic help? YOLO11s at 960, mosaic for 2 vs 10 of 12 epochs

Ultralytics closes mosaic for the last `close_mosaic` = 10 epochs by default, so the 12-epoch runs of
notebook 05 used mosaic in epochs 1–2 only, and their val mAP stopped improving after epoch 1–4 while the
train loss kept falling. Consecutive frames are near-duplicates; a mosaic of 4 random frames shows more
distinct scenes per batch without shrinking objects (each tile keeps its scale, ±50% zoom). Same model,
data and epochs, only `close_mosaic` differs. `yolo11s_960` is reused from notebook 05 when it is in the
same runs folder (Colab Drive), otherwise trained here. The check keeps its own 12-epoch setup,
independent of the config cell.

In [ ]:
if MOSAIC_CHECK:
    train_and_evaluate("yolo11s_960", "yolo11s.pt", 960, 8, 10, epochs=12)
    train_and_evaluate("yolo11s_960_cm2", "yolo11s.pt", 960, 8, 2, epochs=12)

    import matplotlib.pyplot as plt
    check = {"mosaic in epochs 1-2 (close_mosaic 10)": "yolo11s_960",
             "mosaic in epochs 1-10 (close_mosaic 2)": "yolo11s_960_cm2"}
    display(pd.DataFrame({label: metrics(name) for label, name in check.items()}).T.round(4))
    fig, ax = plt.subplots(figsize=(7, 4))
    for label, name in check.items():
        curve = pd.read_csv(RUNS / name / "results.csv")
        curve.columns = curve.columns.str.strip()
        ax.plot(curve["epoch"], curve["metrics/mAP50-95(B)"], marker="o", label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel("Ultralytics val mAP50-95")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RUNS / "mosaic_check.png", dpi=140)
    plt.show()

## 2. Train and evaluate YOLO11m (single frame)

`EPOCHS`, `CLOSE_MOSAIC` and `AUG` from the config, train frames only. Logged to W&B as `yolo11m_1280_e6_aug` (loss and mAP curves, Ultralytics plots,
COCO AP by size, val frames with boxes). Skipped when `WEIGHTS` points to another, already trained model.

In [ ]:
AUG_FLAGS = " ".join(f"--{key.replace('_', '-')} {value}" for key, value in AUG.items())
if WEIGHTS == RUN / "weights" / "last.pt":
    train_and_evaluate(NAME, MODEL, IMGSZ, BATCH, CLOSE_MOSAIC, flags=AUG_FLAGS)
assert WEIGHTS.is_file(), f"no weights at {WEIGHTS}"
print((RUN / "eval.txt").read_text()[-900:] if (RUN / "eval.txt").is_file() else "")

## 3. Predict every val frame

Tracking needs consecutive frames, so all 2846 val frames are predicted (score ≥ 0.01, at most 500 boxes),
in the Kaggle CSV format every detector in the repo writes.

In [ ]:
VAL_INDEX = WORK / "val_index"
VAL_PRED = RUN / "val_all_frames.csv"
!python -m src.analysis.temporal index --split-root {VAL_ROOT} --out {VAL_INDEX}
if not VAL_PRED.is_file():
    !python -m src.models.yolo_baseline predict --weights {WEIGHTS} --test-root {VAL_INDEX} \
        --out {VAL_PRED} --imgsz {IMGSZ} --conf 0.01 --batch 8
print(f"{(time.time() - T0) / 60:.0f} min elapsed")

## 4. Tune the temporal parameters on 4 val clips

Grid over `weight` (share of the track score in a rescored detection) × `max_age` (frames a missed track is
propagated; 0 = rescoring only) without motion compensation, then motion on/off at the best setting.
Selection metric: mAP on the 4 tuning clips. The other 3 clips are not looked at until step 5.

In [ ]:
import argparse, contextlib, io, itertools
import pandas as pd
from pycocotools.coco import COCO

from src.analysis import temporal
from src.models.faster_rcnn import coco_map

parser = argparse.ArgumentParser()
temporal.add_temporal_args(parser)
DEFAULTS = vars(parser.parse_args([]))  # iou 0.5, ema 0.7, new_track 0.3, propagate 0.5, decay 0.8

coco = COCO(str(VAL_ROOT / "annotations_coco.json"))
frames = temporal.read_index(VAL_INDEX / "test_images.csv")
raw = temporal.read_predictions(VAL_PRED)
clips = sorted({row["clip"] for row in frames})
TUNE_CLIPS, TEST_CLIPS = clips[::2], clips[1::2]
ids = lambda clip_set: [int(row["image_id"]) for row in frames if row["clip"] in clip_set]
print("tune:", TUNE_CLIPS, len(ids(TUNE_CLIPS)), "frames\ntest:", TEST_CLIPS, len(ids(TEST_CLIPS)), "frames")


def score(predictions, clip_set):
    detections = [{"image_id": int(image_id), "category_id": int(c) + 1, "score": float(s),
                   "bbox": [x1, y1, x2 - x1, y2 - y1]}
                  for image_id, (classes, scores, boxes) in predictions.items()
                  for c, s, (x1, y1, x2, y2) in zip(classes, scores, boxes.tolist())]
    with contextlib.redirect_stdout(io.StringIO()):  # pycocotools prints a full summary per call
        metrics = coco_map(coco, detections, ids(clip_set))
    return {k: metrics[k] for k in ["mAP", "AP50", "AP_small", "AP_medium", "AP_large"]}


def process(clip_set, **settings):
    clip_frames = [row for row in frames if row["clip"] in clip_set]
    args = argparse.Namespace(**(DEFAULTS | settings))
    return temporal.temporal_rescore(clip_frames, raw, args, VAL_INDEX)


rows = [{"setting": "raw", **score(raw, TUNE_CLIPS)}]
for weight, max_age in itertools.product([0.3, 0.5, 0.7], [0, 3, 6]):
    rows.append({"setting": f"weight {weight}, max_age {max_age}", "weight": weight, "max_age": max_age,
                 "motion": False, **score(process(TUNE_CLIPS, weight=weight, max_age=max_age), TUNE_CLIPS)})
    print(rows[-1]["setting"], f"mAP {rows[-1]['mAP']:.4f}", f"{(time.time() - T0) / 60:.0f} min elapsed")
grid = pd.DataFrame(rows)
best = grid.iloc[1:].sort_values("mAP").iloc[-1]
motion = process(TUNE_CLIPS, weight=best["weight"], max_age=int(best["max_age"]), motion=True)
rows.append({"setting": best["setting"] + ", motion", "weight": best["weight"], "max_age": best["max_age"],
             "motion": True, **score(motion, TUNE_CLIPS)})
grid = pd.DataFrame(rows)
grid.to_csv(RUN / "temporal_grid.csv", index=False)
BEST = grid.iloc[1:].sort_values("mAP").iloc[-1]
SETTINGS = {"weight": float(BEST["weight"]), "max_age": int(BEST["max_age"]), "motion": bool(BEST["motion"])}
print("selected:", SETTINGS)
grid.round(4)

## 5. Raw vs temporal on the 3 held-out clips, by object size

In [ ]:
import matplotlib.pyplot as plt

result = pd.DataFrame([
    {"detector": NAME, "post-processing": "none (single frame)", **score(raw, TEST_CLIPS)},
    {"detector": NAME, "post-processing": f"temporal {SETTINGS}", **score(process(TEST_CLIPS, **SETTINGS), TEST_CLIPS)},
]).set_index(["detector", "post-processing"])
result.loc[(NAME, "gain"), :] = result.iloc[1] - result.iloc[0]
result.to_csv(RUN / "temporal_heldout.csv")
display(result.round(4))

sizes = ["AP_small", "AP_medium", "AP_large"]
ax = result.iloc[:2][sizes].T.plot.bar(figsize=(7, 4), rot=0)
ax.set_ylabel("COCO AP@[.5:.95], held-out val clips")
ax.legend(["single frame", "temporal"])
plt.tight_layout()
plt.savefig(RUN / "temporal_ap_by_size.png", dpi=140)
plt.show()

In [ ]:
# W&B: the tuning grid, the held-out result and the chart in a separate summary run (offline if needed)
try:
    import wandb
    from src.models.faster_rcnn import wandb_run

    run = wandb_run(WANDB_PROJECT, RUNS / f"{NAME}_temporal", config={"weights": str(WEIGHTS), **SETTINGS})
    run.log({"temporal/grid": wandb.Table(dataframe=grid),
             "temporal/heldout": wandb.Table(dataframe=result.reset_index()),
             "temporal/ap_by_size": wandb.Image(str(RUN / "temporal_ap_by_size.png"))})
    run.summary.update({f"heldout/{k}_gain": v for k, v in result.loc[(NAME, "gain")].items()})
    run.finish()
except Exception as error:  # the results are already saved next to the weights
    print(f"W&B summary skipped: {error}")

## 6. Final model: the same settings on train + val

The 7 val clips are scenes the train split does not have, so the submission model also trains on them (every
5th frame, as train). It is not evaluated: its val numbers would be fitted. Sections 2–5 measured the same
recipe trained on train only, and the temporal settings tuned there are reused. Skipped when
`FINAL_TRAIN_VAL` is off or `WEIGHTS` points to another model: the submission then uses `WEIGHTS`.

In [ ]:
FINAL_WEIGHTS = WEIGHTS
if FINAL_TRAIN_VAL and WEIGHTS == RUN / "weights" / "last.pt":
    train_and_evaluate(f"{NAME}_trainval", MODEL, IMGSZ, BATCH, CLOSE_MOSAIC,
                       flags=f"{AUG_FLAGS} --train-val", evaluate=False)
    FINAL_WEIGHTS = RUNS / f"{NAME}_trainval" / "weights" / "last.pt"
assert FINAL_WEIGHTS.is_file(), f"no weights at {FINAL_WEIGHTS}"
print("submission weights:", FINAL_WEIGHTS)

## 7. Test submission with the selected temporal settings

In [ ]:
FINAL_RUN = FINAL_WEIGHTS.parent.parent
TEST_PRED = FINAL_RUN / "submission_raw.csv"
SUBMISSION = FINAL_RUN / "submission_temporal.csv"
if not TEST_PRED.is_file():
    !python -m src.models.yolo_baseline predict --weights {FINAL_WEIGHTS} --test-root {DATA} --out {TEST_PRED} \
        --imgsz {IMGSZ} --conf 0.01 --batch 8
flags = f"--weight {SETTINGS['weight']} --max-age {SETTINGS['max_age']}" + (" --motion" if SETTINGS["motion"] else "")
!python -m src.analysis.temporal apply --pred {TEST_PRED} --images-csv {DATA}/test_images.csv --out {SUBMISSION} {flags}
if ON_KAGGLE:
    !cp {SUBMISSION} /kaggle/working/submission.csv
print(f"done in {(time.time() - T0) / 60:.0f} min")